# Reviewing Genomics score from SnpEff & SIFT4G annotation
EDA for deciding joining method. Finding the Data Science rationale

In [14]:
import cyvcf2
import pandas as pd

In [27]:
def load_sorbi_to_loc(gene_info_path: str) -> dict:
    gi = pd.read_csv(gene_info_path, sep="\t", usecols=["Symbol", "LocusTag"])
    return gi.set_index("LocusTag")["Symbol"].to_dict()


def parse_vcf(vcf_path: str, sorbi_to_loc: dict):
    """
    Returns:
        snpeff_hits : {loc_id: [impact, ...]}   — all SnpEff impacts seen for a gene
        sift_hits   : {loc_id: [sift_score, ...]} — SIFT scores for scored CDS variants
    """
    snpeff_hits: dict[str, list] = {}
    sift_hits:   dict[str, list] = {}

    vcf = cyvcf2.VCF(vcf_path)
    for variant in vcf:

        # ── SnpEff ANN field ───────────────────────────────────────────────
        ann = variant.INFO.get("ANN")
        if ann:
            for entry in str(ann).split(","):
                parts = entry.split("|")
                if len(parts) < 4:
                    continue
                impact   = parts[2]
                sorbi_id = parts[3]
                if impact not in IMPACT_SCORE:
                    continue
                if not sorbi_id.startswith("SORBI_"):
                    continue
                loc_id = sorbi_to_loc.get(sorbi_id)
                if loc_id:
                    snpeff_hits.setdefault(loc_id, []).append(impact)

        # ── SIFT4G SIFTINFO field ──────────────────────────────────────────
        # Format: Allele|Transcript|GeneId|GeneName|Region|VariantType|
        #         RefAA/AltAA|AminoPos|SIFT_score|SIFT_median|NUM_seqs|
        #         Allele_Type|SIFT_prediction
        siftinfo = variant.INFO.get("SIFTINFO")
        if siftinfo:
            for entry in str(siftinfo).split(","):
                parts = entry.split("|")
                if len(parts) < 13:
                    continue
                loc_id    = parts[2]
                sift_pred = parts[12]
                score_str = parts[8]
                if sift_pred not in ("TOLERATED", "DAMAGING"):
                    continue
                try:
                    sift_hits.setdefault(loc_id, []).append(float(score_str))
                except ValueError:
                    pass

    return snpeff_hits, sift_hits

# def build_genomics(snpeff_hits: dict, sift_hits: dict) -> pd.DataFrame:
#     all_genes = set(snpeff_hits) | set(sift_hits)
#     rows = []
#     for gene in all_genes:
#         impacts = snpeff_hits.get(gene, [])
#         scores  = sift_hits.get(gene, [])

#         worst_impact = (
#             min(impacts, key=lambda x: IMPACT_ORDER.index(x))
#             if impacts else "MODIFIER"
#         )
#         impact_score = IMPACT_SCORE[worst_impact]

#         if scores:
#             min_sift      = min(scores)
#             sift_disruption = 1.0 - min_sift
#         else:
#             min_sift        = None
#             sift_disruption = 0.0

#         rows.append({
#             "gene_label":      gene,
#             "worst_impact":    worst_impact,
#             "impact_score":    round(impact_score, 4),
#             "min_sift_score":  round(min_sift, 4) if min_sift is not None else None,
#             "sift_disruption": round(sift_disruption, 4),
#             "genomic_score":   round(impact_score + sift_disruption, 4),
#         })

#     return pd.DataFrame(rows)

def build_genomics(snpeff_hits: dict, sift_hits: dict) -> pd.DataFrame:
    all_genes = set(snpeff_hits) | set(sift_hits)
    rows = []
    for gene in all_genes:
        impacts = snpeff_hits.get(gene, [])
        scores  = sift_hits.get(gene, [])

        worst_impact = (
            min(impacts, key=lambda x: IMPACT_ORDER.index(x))
            if impacts else "MODIFIER"
        )
        impact_score = IMPACT_SCORE[worst_impact]

        if scores:
            min_sift      = min(scores)
            sift_disruption = 1.0 - min_sift
        else:
            min_sift        = None
            sift_disruption = 0.0

        rows.append({
            "gene_label":      gene,
            "worst_impact":    worst_impact,
            "worst_sift_score":  round(min_sift, 4) if min_sift is not None else None,
        })

    return pd.DataFrame(rows)

# def score_genomics


In [16]:
GENE_INFO = "/home/daffa/Work/2026/thesis/resources/NCBI_FTP/gene_info_4558"
VCF_FILE = "/home/daffa/Work/2026/thesis/results/sift4g/SBC10.private.sift4g.vcf.gz"

IMPACT_ORDER = ["HIGH", "MODERATE", "LOW", "MODIFIER"]
IMPACT_SCORE = {"HIGH": 3, "MODERATE": 2, "LOW": 1, "MODIFIER": 0}

# Default parse_vcf output dict

In [17]:
print(f"[ranked_genes] Loading gene ID map …")
sorbi_to_loc = load_sorbi_to_loc(GENE_INFO)

print(f"[ranked_genes] Parsing VCF: {VCF_FILE}")
snpeff_hits, sift_hits = parse_vcf(VCF_FILE, sorbi_to_loc)

[ranked_genes] Loading gene ID map …
[ranked_genes] Parsing VCF: /home/daffa/Work/2026/thesis/results/sift4g/SBC10.private.sift4g.vcf.gz


In [18]:
snpeff_hits

{'LOC8081570': ['MODIFIER', 'MODIFIER', 'MODIFIER', 'MODIFIER'],
 'LOC8059226': ['MODIFIER'],
 'LOC8059546': ['MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER'],
 'LOC8059547': ['MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFIER',
  'MODIFI

In [19]:
sift_hits

{'LOC8061378': [0.63],
 'LOC8061396': [1.0, 1.0, 1.0, 0.6, 1.0],
 'LOC8060857': [0.54, 1.0, 0.93, 0.52, 1.0, 0.49],
 'LOC8060858': [0.58],
 'LOC110431735': [1.0],
 'LOC8060870': [0.96],
 'LOC8060872': [1.0, 0.1],
 'LOC8060873': [1.0],
 'LOC8080155': [0.06],
 'LOC8060878': [1.0],
 'LOC8060882': [1.0],
 'LOC8060883': [0.43],
 'LOC8060884': [0.14],
 'LOC8060885': [1.0,
  1.0,
  1.0,
  1.0,
  0.29,
  1.0,
  1.0,
  0.62,
  1.0,
  0.94,
  1.0,
  0.58,
  0.47,
  1.0],
 'LOC8080164': [1.0, 0.9],
 'LOC8079645': [1.0],
 'LOC8080170': [1.0, 1.0, 1.0],
 'LOC110433544': [0.51, 1.0, 1.0],
 'LOC8079688': [0.06],
 'LOC8079693': [1.0],
 'LOC110436660': [1.0],
 'LOC8080211': [1.0],
 'LOC8080217': [1.0, 1.0],
 'LOC8080218': [1.0],
 'LOC8079703': [0.06, 0.18, 0.49],
 'LOC8080221': [1.0],
 'LOC8079705': [0.85, 0.17, 1.0, 1.0, 0.23],
 'LOC8080223': [0.06, 1.0],
 'LOC8057037': [0.08],
 'LOC8057046': [0.71, 0.16, 0.05, 0.47],
 'LOC110431853': [1.0],
 'LOC110432034': [1.0],
 'LOC8081201': [0.17,
  1.0,
  0.81,

In [28]:
df = build_genomics(snpeff_hits, sift_hits)

In [29]:
df

,gene_label,worst_impact,worst_sift_score
0,LOC8064487,MODIFIER,NaN
1,LOC8071365,MODIFIER,NaN
2,LOC8072794,MODERATE,0.06
3,LOC8060810,MODIFIER,NaN
4,LOC110430835,MODIFIER,NaN
...,...,...,...
17228,LOC8070023,MODIFIER,NaN
17229,LOC8078479,MODIFIER,NaN
17230,LOC8059683,MODERATE,0.21
17231,LOC110436877,MODIFIER,NaN
